# Chapter 39 - The Recommendation and What Happened Next

This notebook closes the Sedge Shoes case study by recomputing the numbers behind Chapter 39's four decisions from the calibrated case-study model: hold the £6 million budget, move about £250,000 out of Meta and £100,000 out of Search, read the Meta cut as a judgement about the next pound, and choose the next experiments by their value of information.
The six-month refresh at the end of the chapter is narrative: the synthetic data stop at week 155, so it has no counterpart here.

In [1]:
import numpy as np
import pandas as pd

from mmm_book import RANDOM_SEED
from mmm_book.case_model import fit_case_model, posterior_curves, prepare_case_data
from mmm_book.optimise import BOOK_PLAN_ANNUAL, band, greedy_allocate
from mmm_book.sedge import CHANNELS, correct_distribution, generate_sedge

pd.set_option("display.width", 120)
WEEKS = 52
fixed = correct_distribution(generate_sedge(seed=RANDOM_SEED))
data = prepare_case_data(fixed, "model2")
idata = fit_case_model(fixed, "model2", calibrate=True)
curves = posterior_curves(idata, data)
plan = {ch: BOOK_PLAN_ANNUAL[ch] / WEEKS for ch in CHANNELS}

## Decisions one and two: hold the budget, reallocate

The reallocation's gain at the posterior mean, its range across draws, and the probability that it beats the current mix.

In [2]:
gain = curves.gain_draws(plan) * WEEKS
lo, hi = np.quantile(gain, [0.05, 0.95])
print(
    f"expected gain {gain.mean():,.0f} a year (90% {lo:,.0f} to {hi:,.0f}); "
    f"profit {0.5 * gain.mean():,.0f}"
)
print(f"probability the plan beats the current mix: {(gain > 0).mean():.3f}")
print(f"half a year: {gain.mean() / 2:,.0f} of revenue, {gain.mean() / 4:,.0f} of profit")

expected gain 266,190 a year (90% 109,911 to 436,515); profit 133,095
probability the plan beats the current mix: 0.999
half a year: 133,095 of revenue, 66,547 of profit


The probability is close to one because every draw agrees on the direction of the moves; the size is uncertain.
It covers only the model: not a future unlike the past, changes in media costs, or errors the model cannot see.

## Decision three: Meta

Meta's average and marginal return, from the same draws.

In [3]:
meta_avg = idata.posterior["roas_meta"].to_numpy().ravel()
meta_marg = curves.marginal_draws("meta")
print(
    f"Meta average {meta_avg.mean():.2f}, marginal {meta_marg.mean():.2f} "
    f"(90% {np.quantile(meta_marg, 0.05):.2f} to {np.quantile(meta_marg, 0.95):.2f})"
)
print(f"cut: {(1 - BOOK_PLAN_ANNUAL['meta'] / 1.5e6):.0%}")

Meta average 1.65, marginal 1.12 (90% 0.63 to 1.59)
cut: 17%


## Decision four: where to measure next

For each channel, take the tenth of posterior draws in which its next pound is worth least and the tenth in which it is worth most, rerun the optimiser on each set with every channel free to move by 20 per cent, and measure how much money changes channel between the two answers.

In [4]:
lo20, hi20 = band(curves, 0.20)
rows = {}
for ch in CHANNELS:
    m = curves.marginal_draws(ch)
    p10, p90 = np.quantile(m, [0.10, 0.90])
    low = greedy_allocate(curves.subset(m <= p10), lo20, hi20)
    high = greedy_allocate(curves.subset(m >= p90), lo20, hi20)
    moved = sum(abs(low[c] - high[c]) for c in CHANNELS) * WEEKS / 1e6 / 2
    rows[ch] = {
        "marginal p10": p10,
        "marginal p90": p90,
        "budget moved (m)": moved,
        **{f"low {c}": low[c] * WEEKS / 1e6 for c in CHANNELS},
        **{f"high {c}": high[c] * WEEKS / 1e6 for c in CHANNELS},
    }
voi = pd.DataFrame(rows).T.sort_values("budget moved (m)", ascending=False)
print(voi[["marginal p10", "marginal p90", "budget moved (m)"]].round(2))
plans = voi[[c for c in voi.columns if c.startswith(("low", "high"))]]
print(plans.loc[["meta", "youtube"]].round(2))

         marginal p10  marginal p90  budget moved (m)
meta             0.76          1.49              0.54
youtube          1.20          2.55              0.35
search           1.09          1.68              0.32
ooh              1.10          1.68              0.32
tv               2.14          2.30              0.09
         low tv  low search  low meta  low youtube  low ooh  high tv  high search  high meta  high youtube  high ooh
meta       1.68        1.36      1.20         0.96     0.80     1.68         1.22       1.74          0.64      0.72
youtube    1.68        1.31      1.53         0.64     0.84     1.68         1.34       1.20          0.96      0.82


Meta and YouTube rank first and second, and for the same reason: the plan's choice between them turns on how their shared paid-social effect is split.
Television ranks last, because its curve is the best known.
Meta has just been tested, with a holdout that measured its average return rather than its slope, so the next experiments are a YouTube geo test, which separates the pair from the other side, and then a Meta test that raises spend.

## Takeaway

The recommendation is narrower than "Television is the best channel", and more defensible for being narrower.
At Sedge's current spend levels the next pound buys more from Television than from Search or Meta in every posterior draw, and moving a moderate amount of budget that way is worth about £0.27 million a year, with a range the board can see.
A recommendation built this way earns a phased rollout, a monitoring plan and a next experiment.